# 13 — String Operations

The `str` accessor unlocks 40+ vectorized string operations — all faster than
Python loops and fully NaN-aware. This notebook covers every method you'll use
in data cleaning, feature engineering, and NLP preprocessing.

In [ ]:
import pandas as pd
import numpy as np

# Primary dataset: customer reviews and product data
np.random.seed(42)

customers = pd.DataFrame({
    'customer_id': range(1001, 1021),
    'name':        ['Alice Smith', 'Bob  Jones', '  Carol White ', 'dave brown', 'EVE JOHNSON',
                    'Frank Lee', 'Grace Kim', 'Hank O\'Brien', 'Ivy Chen', 'Jack Das',
                    'Kate Patel', 'Leo Singh', 'Maya Gupta', 'Nate Kumar', 'Olivia Nair',
                    'Paul Mehta', 'Quinn Rao', 'Rose Iyer', 'Sam Shah', 'Tina Das'],
    'email':       [f'user{i}@example.com' for i in range(20)],
    'phone':       ['(98) 7654-3210', '+91-876-543-2109', '765 432 1098', '6543210987',
                    None, '543 210 9876', '43 21 09 87 65', '3210987654', '2109876543', '1098765432',
                    '(09) 8765-4321', '987654321', '8765432101', '7654321012', '6543210123',
                    '5432101234', '4321012345', '3210123456', '2101234567', '1012345678'],
    'city':        np.random.choice(['Mumbai', 'Delhi', 'Bangalore', 'Pune', 'Chennai'], 20),
    'tags':        ['premium,loyal,high-value', 'new,trial', 'loyal,discount', 'at-risk',
                    'premium,corporate', 'new', 'loyal', 'premium,loyal', 'at-risk,discount',
                    'new,trial', 'corporate', 'premium', 'loyal,high-value', 'new', 'at-risk',
                    'premium,loyal', 'new,corporate', 'loyal', 'at-risk,discount', 'premium'],
    'notes':       ['Excellent product! Highly recommend.', 'Not bad, price too high.',
                    'Great quality. Will buy again!', 'Poor packaging, product OK.',
                    'AMAZING! 5 stars.', 'Decent product. Average quality.',
                    'Good value for money.', 'Disappointed. Expected better.',
                    'Super fast delivery!', 'Product as described.',
                    'Could be better. 3/5', 'Excellent service!',
                    'Loved it! Premium quality.', 'Ok product, slow delivery.',
                    'Very satisfied!', 'Needs improvement.', 'Worth the price.',
                    'Not recommended.', 'Great purchase!', 'Average.']
})

customers.head(5)

## 1. Case Operations

In [ ]:
names = customers['name']

print("upper():  ", names.str.upper().head(3).tolist())
print("lower():  ", names.str.lower().head(3).tolist())
print("title():  ", names.str.title().head(3).tolist())
print("capitalize():", names.str.capitalize().head(3).tolist())
print("swapcase():",  names.str.swapcase().head(3).tolist())

## 2. Whitespace and Padding

In [ ]:
# strip, lstrip, rstrip
messy = pd.Series(['  Alice  ', 'Bob  ', '  Carol'])
print(messy.str.strip())    # both ends
print(messy.str.lstrip())   # left only
print(messy.str.rstrip())   # right only

# Normalize all names at once
customers['name_clean'] = customers['name'].str.strip().str.title()
print("\nCleaned names:")
print(customers['name_clean'].head(5).tolist())

In [ ]:
# pad() — pad to fixed width
ids = pd.Series([1, 12, 123, 1234, 12345])
padded = ids.astype(str).str.pad(width=6, fillchar='0', side='left')
print(padded)  # zero-padded IDs like zfill

## 3. Searching — contains, startswith, endswith

In [ ]:
notes = customers['notes']

# contains() — regex by default
positive = notes.str.contains(r'excellent|great|amazing|love', case=False, regex=True, na=False)
print(f"Positive reviews: {positive.sum()}")
print(customers.loc[positive, ['name_clean', 'notes']].head())

In [ ]:
emails = customers['email']

# startswith / endswith — literal string match (faster than regex for simple checks)
print(emails.str.startswith('user1').value_counts())
print(emails.str.endswith('.com').all())  # all end with .com

## 4. len(), count(), find()

In [ ]:
# len() — character count per element
print("Review lengths:")
print(notes.str.len().describe())

# count() — count occurrences of substring/pattern
exclamation = notes.str.count('!')
print("\nExclamation marks per review:")
print(exclamation.value_counts())

# find() — position of first occurrence (-1 if not found)
print("\nPosition of 'product':")
print(notes.str.find('product').head())

## 5. replace() — String Substitution

In [ ]:
# str.replace() — vectorized, supports regex
# Clean phone numbers: remove all non-digits
phones = customers['phone'].dropna()
clean_phones = phones.str.replace(r'\D', '', regex=True)
print("Cleaned phones:")
print(clean_phones.head(10).tolist())

In [ ]:
# Multiple substitutions — chain or use a loop
urls = pd.Series([
    'https://www.example.com/product?id=123&ref=home',
    'http://example.com/product?id=456',
    'https://example.com/category/electronics'
])

# Remove protocol and trailing params
clean_urls = (
    urls
    .str.replace(r'https?://(www\.)?', '', regex=True)
    .str.replace(r'\?.*$', '', regex=True)  # remove query string
)
print(clean_urls)

## 6. split() and str[n] Slicing

In [ ]:
# split() — returns list of parts per element
names_split = customers['name_clean'].str.split(' ')
print(names_split.head())

# expand=True — splits into separate columns
name_parts = customers['name_clean'].str.split(' ', expand=True)
name_parts.columns = ['first_name', 'last_name']
print(name_parts.head())

In [ ]:
# n: max splits
path = pd.Series(['/users/alice/docs/file.pdf', '/home/bob/data.csv', '/root/config.yaml'])
parts = path.str.split('/', n=3, expand=True)
print(parts)

# rsplit — split from right
filenames = path.str.rsplit('/', n=1, expand=True)
filenames.columns = ['directory', 'filename']
print(filenames)

In [ ]:
# String slicing with str[start:stop:step]
codes = pd.Series(['PROD-2023-001', 'PROD-2024-002', 'SRVS-2023-003', 'PROD-2022-004'])

print(codes.str[:4])     # type prefix
print(codes.str[5:9])    # year
print(codes.str[-3:])    # last 3 = sequential number

## 7. extract() and extractall()

In [ ]:
# extract() — first match, one row per element
# Use named groups for readable column names
reviews = pd.Series([
    'Rated 4/5 stars. Great product.',
    'I give this 3 out of 5.',
    '5 stars! Excellent!',
    'No rating given.',
    '2.5/5 — disappointing.'
])

ratings_extracted = reviews.str.extract(
    r'(?P<rating>[\d\.]+)\s*(?:/\s*5|stars|out of 5)',
    expand=True
)
print(ratings_extracted)

In [ ]:
# extractall() — ALL matches, multi-level index result
text = pd.Series([
    'Order 1001 and 1002 shipped',
    'Order 2001 confirmed',
    'No orders'
])

all_orders = text.str.extractall(r'(\d+)')
all_orders.columns = ['order_id']
print(all_orders)

## 8. cat() — String Concatenation

In [ ]:
# cat() concatenates elements of a Series
cities = pd.Series(['Mumbai', 'Delhi', 'Bangalore', np.nan, 'Chennai'])

# Join all non-NaN elements
print(cities.str.cat(sep=', ', na_rep='Unknown'))

# Concatenate two Series element-wise
first = pd.Series(['Alice', 'Bob', 'Carol'])
last  = pd.Series(['Smith', 'Jones', 'White'])
full_name = first.str.cat(last, sep=' ')
print(full_name)

## 9. Multi-Value Column — Tags Parsing

In [ ]:
# Real-world: tags stored as comma-separated strings
# Convert to one-hot encoded boolean columns

tags_series = customers['tags'].str.split(',').explode()  # one tag per row
print("All tag values:")
print(tags_series.value_counts())

In [ ]:
# One-hot encode tags — str.get_dummies()
tag_dummies = customers['tags'].str.get_dummies(sep=',')
print(tag_dummies.head())
print(f"\nShape: {tag_dummies.shape}")

## 10. wrap() and center() — Text Formatting

In [ ]:
# wrap() — word-wrap long strings to fixed width
long_text = pd.Series([
    'This is a very long product description that needs to be wrapped nicely.',
    'Short desc.'
])
print(long_text.str.wrap(40))

## 11. Production — Email Validation Pattern

In [ ]:
emails_raw = pd.Series([
    'alice@company.com',
    'bob.jones@mail.co.uk',
    'invalid-email',
    'carol@',
    'dave@company',
    'eve.smith+tag@domain.org',
    'frank@123.com',
])

# Simple email validation
email_pattern = r'^[a-zA-Z0-9._%+\-]+@[a-zA-Z0-9.\-]+\.[a-zA-Z]{2,}$'
valid_emails = emails_raw.str.match(email_pattern)

result = pd.DataFrame({
    'email': emails_raw,
    'valid': valid_emails
})
print(result)

## 12. NaN Safety — na= parameter

In [ ]:
s = pd.Series(['hello', None, 'world', np.nan, 'pandas'])

# Default: NaN in → NaN out (propagates)
print(s.str.upper())

# na=False: NaN rows return False for bool methods
print("\ncontains with na=False:")
print(s.str.contains('o', na=False))

# na=True: NaN rows return True for bool methods
print("\ncontains with na=True:")
print(s.str.contains('o', na=True))

## 13. Quick Reference — str Accessor Methods

| Method | Description |
|--------|-------------|
| `.str.lower()` / `.upper()` / `.title()` | Case conversion |
| `.str.strip()` / `.lstrip()` / `.rstrip()` | Whitespace removal |
| `.str.contains(pat, regex=True)` | Boolean membership check |
| `.str.startswith(pat)` / `.endswith(pat)` | Prefix/suffix check |
| `.str.replace(pat, repl, regex=True)` | Substitution |
| `.str.split(sep, expand=True)` | Split to columns |
| `.str.extract(r'pattern')` | Regex group capture |
| `.str.extractall(r'pattern')` | All regex matches |
| `.str.len()` | String length |
| `.str.count(pat)` | Count occurrences |
| `.str.get_dummies(sep)` | One-hot from delimited column |
| `.str.cat(sep=', ')` | Concatenate elements |
| `.str.pad(width, side, fillchar)` | Pad to fixed width |
| `.str[start:stop]` | Slice characters |